**Tabular Dataset Integration**
- **Objective:** Wrap a Pandas DataFrame into a PyTorch-compatible format.
- **Task:** Subclass torch.utils.data.Dataset.
- **Action:**
- 1. Load a familiar tabular dataset (like the California Housing dataset) using Pandas.
- 2. Create a custom Dataset class where __init__ takes the DataFrame and converts the features and targets to PyTorch tensors (torch.tensor(df.values, dtype=torch.float32)).
- 3. Implement __len__ to return the number of rows.
- 4. Implement __getitem__ to return a (features, target) tuple for a given index.
- 5. Wrap the dataset in a DataLoader with batch_size=32 and shuffle=True, and iterate through one epoch.

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import fetch_california_housing
import pandas as pd

# 1. Load the California Housing dataset into Pandas
california = fetch_california_housing(as_frame=True)
df = california.frame
target_col = california.target_names[0]  # 'MedHouseVal'
feature_cols = california.feature_names

# 2. Subclass torch.utils.data.Dataset
class TabularDataset(Dataset):
    def __init__(self, dataframe, feature_columns, target_column):
        # Extract features and targets
        features = dataframe[feature_columns].values
        targets = dataframe[target_column].values
        
        # Convert to PyTorch tensors (float32)
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(targets, dtype=torch.float32).unsqueeze(1) # Shape: [N, 1]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 3. Instantiate the Custom Dataset
dataset = TabularDataset(df, feature_cols, target_col)

# 4. Wrap in a DataLoader
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# 5. Iterate through one epoch
for batch_idx, (X_batch, y_batch) in enumerate(dataloader):
    # Print the shapes of the first batch to verify
    if batch_idx == 0:
        print(f"Batch Count: {len(dataloader)}")
        print(f"Features shape: {X_batch.shape}")
        print(f"Targets shape: {y_batch.shape}")
        break  # Truncated after 1 batch for demonstration

Batch Count: 645
Features shape: torch.Size([32, 8])
Targets shape: torch.Size([32, 1])


**Variable Length Sequences and Custom Collate Functions**
- **Objective:** Handle non-uniform data structures (common in NLP or time-series).
- **Task:** DataLoaders expect uniform tensor sizes to stack them into batches. Write a custom collate_fn.
- **Action:**
- 1. Create a dummy dataset that returns 1D tensors (sequences) of random varying lengths.
- 2. Write a custom collate_fn function that takes a list of (sequence, label) tuples.
- 3. Inside the collate_fn, use torch.nn.utils.rnn.pad_sequence to pad the sequences with zeros so they all match the length of the longest sequence in that specific batch.
- 4. Pass this function to the DataLoader(..., collate_fn=my_collate) and verify your batches output uniform 2D padded tensors.

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import random

# 1. Create a dummy dataset returning varying length 1D tensors
class VariableLengthDataset(Dataset):
    def __init__(self, num_samples=100, min_len=5, max_len=15):
        self.num_samples = num_samples
        self.data = []
        
        for _ in range(num_samples):
            # Generate a random length for the sequence
            seq_len = random.randint(min_len, max_len)
            # Create a 1D sequence tensor with random integers
            sequence = torch.randint(low=1, high=100, size=(seq_len,))
            # Create a dummy scalar label
            label = torch.tensor(random.choice([0, 1]), dtype=torch.float32)
            self.data.append((sequence, label))

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.data[idx]

# 2. Write a custom collate function
def pad_collate_fn(batch):
    # batch is a list of tuples: [(seq1, label1), (seq2, label2), ...]
    sequences = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    
    # Pad sequences to match the longest sequence in this specific batch
    # batch_first=True outputs shape: [batch_size, max_seq_len]
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
    
    # Stack individual scalar labels into a single tensor
    stacked_labels = torch.stack(labels).unsqueeze(1)
    
    return padded_sequences, stacked_labels

# 3. Instantiate dataset and DataLoader with the custom collate function
dataset = VariableLengthDataset(num_samples=10, min_len=3, max_len=10)
dataloader = DataLoader(dataset, batch_size=3, shuffle=True, collate_fn=pad_collate_fn)

# 4. Verify the batches output uniform 2D padded tensors
for batch_idx, (seq_batch, label_batch) in enumerate(dataloader):
    print(f"Batch {batch_idx + 1}:")
    print(f"  Sequences Shape: {seq_batch.shape}")
    print(f"  Labels Shape:    {label_batch.shape}")
    print(f"  Padded Batch Tensor:\n{seq_batch}\n")

Batch 1:
  Sequences Shape: torch.Size([3, 10])
  Labels Shape:    torch.Size([3, 1])
  Padded Batch Tensor:
tensor([[14, 90, 76,  3, 88, 86, 25,  0,  0,  0],
        [63, 58, 56, 34, 15, 42, 51, 39,  0,  0],
        [87, 73, 22, 24, 82, 50, 57, 27, 12, 71]])

Batch 2:
  Sequences Shape: torch.Size([3, 7])
  Labels Shape:    torch.Size([3, 1])
  Padded Batch Tensor:
tensor([[44, 12, 80, 18, 95, 31,  0],
        [44, 76, 36, 31, 75, 13,  3],
        [60, 76,  7,  0,  0,  0,  0]])

Batch 3:
  Sequences Shape: torch.Size([3, 7])
  Labels Shape:    torch.Size([3, 1])
  Padded Batch Tensor:
tensor([[49, 86, 81, 37, 55, 20,  0],
        [87, 12, 57, 42, 49, 32, 94],
        [54, 93, 69,  3, 98, 63,  0]])

Batch 4:
  Sequences Shape: torch.Size([1, 7])
  Labels Shape:    torch.Size([1, 1])
  Padded Batch Tensor:
tensor([[24, 35, 89, 83, 26, 16, 46]])

